# 16 · Full metric set for the P1/P2 forecasts (D30)

RMSE stays the headline metric. This notebook shows every regression and classification metric of decision D30 for the forecasts behind paper Tables P1 and P2 (S1 and S6). The metrics are defined in `src/metrics.py` and `docs/DECISIONS.md` D30.

- **Regression:** MSE, RMSE, MAE, MedAE, bias, MAPE, NRMSE, R² (SOH level) and R²_Δ (SOH change since the origin).
- **Reuse class (C0–C3):** accuracy, balanced accuracy, macro precision / recall / specificity / F1, MCC, Cohen's κ (unweighted and linear-weighted), confusion matrices.
- **End of first life (SOH ≤ 80, positive):** precision, recall, specificity, F1, MCC, AUC-ROC.
- **Log loss** is not computed: every forecaster gives point forecasts, not class probabilities.

**Where the numbers come from.** `scripts/run_d30.py` re-runs the forecasts with the committed hyperparameters and writes `results/forecasting_d30_<dataset>.json`. It writes nothing unless the recomputed RMSE reproduces the committed P1/P2 values. This notebook only reads those files and needs no raw data.

SOH is relative to the first observed check, not nominal capacity.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "data_pipeline.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
RESULTS_DIR = REPO_ROOT / "results"
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

DATASETS = {"s1": "S1 (Stanford second life, unseen battery)", "s6": "S6 (Wheeler LFP, unseen usage condition)"}
ORDER = ["persistence", "linear_extrapolation", "ridge_schedule", "ridge_F0", "ridge_F1", "gbm_F1", "mlp", "tft"]
CLASS_NAMES = ["C0", "C1", "C2", "C3"]

res = {}
for k in DATASETS:
    path = RESULTS_DIR / f"forecasting_d30_{k}.json"
    if path.exists():
        r = json.loads(path.read_text())
        res[k] = r
        env = r["environment"]
        print(f"{k}: {path.name} | commit {env['git_commit'][:7]} dirty={env['git_dirty']} | profile {r['profile']} | "
              f"reproduction {r['reproduction_check'].get('passed')} (max abs diff {r['reproduction_check'].get('max_abs_diff')})")
    else:
        print(f"{k}: {path.name} not found - run `python scripts/run_d30.py --dataset {k}` first")


def table(r, key):
    d = pd.DataFrame(r[key])
    return d.set_index("method").reindex([m for m in ORDER if m in set(d["method"])])

## 1. Measured class support

These are the measured reuse-class counts of all samples, per group. A metric that needs two classes (MCC, κ, AUC) is undefined where only one class occurs.

In [ ]:
for k, r in res.items():
    print(DATASETS[k])
    display(pd.DataFrame(r["class_support"], index=CLASS_NAMES).T)

## 2. Regression metrics, P2 (mean over held-out groups)

The RMSE column equals paper Table P2. Bias = mean(forecast − measured), so a positive bias means SOH is overestimated (fade underestimated).

In [ ]:
REG = ["reg_RMSE", "reg_MSE", "reg_MAE", "reg_MedAE", "reg_Bias", "reg_NRMSE", "reg_MAPE", "reg_R2", "reg_R2_change"]
for k, r in res.items():
    print(DATASETS[k])
    t = table(r, "p2_over_groups")
    display(t[REG].rename(columns=lambda c: c.removeprefix("reg_")).round(3))

### Pooled over all held-out samples (mean over seeds)

Pooled values weight groups by their number of samples, so they differ from the mean over groups.

In [ ]:
for k, r in res.items():
    print(DATASETS[k])
    t = table(r, "p2_pooled")
    display(t[REG].rename(columns=lambda c: c.removeprefix("reg_")).round(3))

## 3. Classification metrics, P2 (all held-out samples pooled, mean over seeds)

In [ ]:
CLS = ["cls_accuracy", "cls_balanced_accuracy", "cls_precision_macro", "cls_recall_macro", "cls_specificity_macro",
       "cls_f1_macro", "cls_mcc", "cls_kappa", "cls_kappa_linear"]
EOL = ["eol_accuracy", "eol_balanced_accuracy", "eol_precision", "eol_recall", "eol_specificity", "eol_f1", "eol_mcc", "eol_auc_roc"]
for k, r in res.items():
    t = table(r, "p2_pooled")
    print(DATASETS[k], "- reuse class C0-C3")
    display(t[CLS].rename(columns=lambda c: c.removeprefix("cls_")).round(3))
    print(DATASETS[k], "- end of first life (SOH <= 80 positive)")
    display(t[EOL].rename(columns=lambda c: c.removeprefix("eol_")).round(3))

### Confusion matrices (seed 0, pooled)

In [ ]:
SHOW = ["persistence", "linear_extrapolation", "gbm_F1", "tft"]
for k, r in res.items():
    t = table(r, "p2_pooled")
    methods = [m for m in SHOW if m in t.index]
    fig, axes = plt.subplots(1, len(methods), figsize=(3.0 * len(methods), 3.0), sharey=True)
    for ax, m in zip(np.atleast_1d(axes), methods):
        cm = np.asarray(t.loc[m, "confusion"], float)
        norm = cm / np.where(cm.sum(1, keepdims=True) > 0, cm.sum(1, keepdims=True), 1)
        ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
        for (i, j), v in np.ndenumerate(cm):
            ax.text(j, i, int(v), ha="center", va="center", fontsize=9, color="white" if norm[i, j] > 0.6 else "black")
        ax.set_xticks(range(4), CLASS_NAMES); ax.set_yticks(range(4), CLASS_NAMES)
        ax.set_xlabel("Forecast class"); ax.set_title(m, fontsize=10)
    np.atleast_1d(axes)[0].set_ylabel("Measured class")
    fig.suptitle(f"{k.upper()}, P2: reuse-class confusion (counts; shading = share of the measured class)", fontsize=10)
    plt.tight_layout(); plt.show()

## 4. Per held-out group (mean over seeds)

In [ ]:
for k, r in res.items():
    g = pd.DataFrame(r["p2_per_group"])
    print(DATASETS[k])
    for col in ("reg_MAE", "reg_Bias", "reg_R2_change"):
        display(g.pivot(index="method", columns="Battery_ID", values=col).reindex(ORDER).dropna(how="all")
                 .round(3).rename_axis(col.removeprefix("reg_")))

In [ ]:
fig, axes = plt.subplots(2, len(res), figsize=(6 * max(len(res), 1), 6), squeeze=False)
for c, (k, r) in enumerate(res.items()):
    g = pd.DataFrame(r["p2_per_group"])
    names = sorted(g["Battery_ID"].unique(), key=str)
    x = np.arange(len(names))
    for i, (m, colour) in enumerate((("linear_extrapolation", "#8a8981"), ("gbm_F1", "#1baf7a"), ("tft", "#2a78d6"))):
        sub = g[g["method"] == m].set_index("Battery_ID").reindex(names)
        for row, (col, lab) in enumerate((("reg_MAE", "MAE (pp)"), ("reg_Bias", "Bias (pp)"))):
            axes[row, c].bar(x + (i - 1) * 0.27, sub[col], 0.27, color=colour, label=m)
            axes[row, c].set_ylabel(lab); axes[row, c].set_xticks(x, names)
    axes[1, c].axhline(0, color="black", linewidth=0.8); axes[1, c].set_xlabel("Held-out group")
    axes[0, c].set_title(f"{k.upper()}, P2", loc="left"); axes[0, c].legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

## 5. P1 (final step of the continual stream), mean over groups

In [ ]:
for k, r in res.items():
    print(DATASETS[k], "(continual methods included:", r["p1_includes_continual_methods"], ")")
    t = pd.DataFrame(r["p1_over_groups"]).set_index("method")
    display(t[[c for c in REG + CLS if c in t]].rename(columns=lambda c: c.split("_", 1)[1]).round(3))

## 6. Reading the metrics

- **R² vs R²_Δ.** Level R² is high for every method, persistence included, because SOH differs between origins. R²_Δ is the informative one: it is ≤ 0 for persistence and shows how much of the actual fade a method forecasts.
- **MAPE** is close to MAE / SOH, because SOH stays near 100 %. It adds little beyond MAE and is kept for completeness.
- **Classification** depends on how many class boundaries the fade crosses in the test data. In S1 almost all measured SOH is in C0, so S1 class metrics are descriptive at most. The S6 values are the informative ones.
- **AUC** ranks the forecast SOH against the measured end-of-first-life label. It is not a calibrated probability.

The tables and figures for the paper come from `scripts/make_d30_outputs.py` (`paper/tables/d30_*.tex`, `paper/figures/d30_*.png`).